In [1]:
import sys
import time
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

sys.path.append(str(Path.cwd().parent))
from scripts.evaluation import evaluate

RANDOM_STATE = 25
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'

X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
X_test  = pd.read_parquet(DATA_DIR / 'X_test.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']
y_test  = pd.read_parquet(DATA_DIR / 'y_test.parquet') ['Victims_Condition']

numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

dt_full = Pipeline([
    ('pre', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

print("Fitting unrestricted DT on full training set...")
t0 = time.time()
dt_full.fit(X_train, y_train)
y_pred_test = dt_full.predict(X_test)
print(f"  fit + predict: {time.time() - t0:.1f}s\n")

test_result = evaluate(y_test, y_pred_test, model_name='DecisionTree (unrestricted, TEST)')

dt_capped = Pipeline([
    ('pre', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Running 5-fold CV with max_depth=20...")
t0 = time.time()
capped_scores = cross_val_score(dt_capped, X_train, y_train,
                                cv=cv, scoring='f1_macro', n_jobs=-1)
elapsed = time.time() - t0

print(f"\n  CV mean = {capped_scores.mean():.4f} (± {capped_scores.std():.4f})")
print(f"  Phase 2 unrestricted was:  0.4353 (± 0.0016)")
print(f"  Time: {elapsed:.1f}s")

Fitting unrestricted DT on full training set...
  fit + predict: 229.7s


DecisionTree (unrestricted, TEST)
Accuracy : 0.6251
Macro-F1 : 0.4397
MCC      : 0.1529
  F1 [With dead victims]: 0.2166
  F1 [With injured victims]: 0.7497
  F1 [Without victims]: 0.3527

Confusion matrix:
                           pred_With dead victims  pred_With injured victims  \
true_With dead victims                       1403                       3985   
true_With injured victims                    4370                      49309   
true_Without victims                          913                      11815   

                           pred_Without victims  
true_With dead victims                      881  
true_With injured victims                 12757  
true_Without victims                       7184  
Running 5-fold CV with max_depth=20...

  CV mean = 0.4280 (± 0.0041)
  Phase 2 unrestricted was:  0.4353 (± 0.0016)
  Time: 54.4s
